
# Voxel features

Clustering uses the voxel field you define — not a fixed T1 image.
This page is the intensity pair: ``raw`` (concatenate modality intensities
inside the ROI) and ``concat`` (join families column-wise).

Custom formulas and plugins live on the next page. Texture maps follow
that. This page is only ``raw`` vs ``concat``.


Load two demo subjects. ``raw`` concatenates LAP and PVP intensities
inside the ROI.



In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from habit.contracts import cohort_from_directory
from habit.datasets import fetch_demo
from habit.pipeline.assembly import build_habitat_components
from habit.spec import HabitatSpec, Spec
from habit.viz import plot_habitat_overlay
import habit.recipes as recipes

DATA = fetch_demo()
MODALITIES = ("LAP", "PVP")
ROI = "LAP"
cohort = cohort_from_directory(DATA, modalities=MODALITIES, roi=ROI)[:2]
subject = cohort[0]
print(f"Cohort: {len(cohort)} subjects -> {list(cohort.subject_ids)}")

``raw(modalities)``: one column per series. Print the first voxel rows
so you can see the intensities clustering will use (before any
feature preprocessor).



In [ ]:
raw_spec = HabitatSpec(
    name="route_raw",
    voxel_feature_extractor=Spec("raw", {"modalities": list(MODALITIES)}),
    supervoxelizer=Spec("kmeans", {"n_supervoxels": 8, "n_init": 3}),
    habitat_model_fitter=Spec(
        "kmeans",
        {"min_habitats": 2, "max_habitats": 3, "validation": "elbow", "n_init": 3},
    ),
    habitat_assigner=Spec("nearest_centroid"),
    habitat_features=(Spec("volume"), Spec("msi"), Spec("ith_score")),
    random_seed=11,
)
raw_field = (
    build_habitat_components(raw_spec)
    .pipeline(assigner=None)
    .voxel_feature_extractor(subject)
)
print("raw feature table:")
print(raw_field.feature_frame().head())
raw_field.feature_frame().head()

The matrix clustering sees: one row per ROI voxel, one column per
modality (rows subsampled so the heatmap stays legible).



In [ ]:
frame = raw_field.feature_frame()
row_step = max(1, len(frame) // 60)
sample = frame.iloc[::row_step]
fig_matrix, ax_matrix = plt.subplots(figsize=(4.6, 4.6), constrained_layout=True)
im = ax_matrix.imshow(sample.to_numpy(dtype=float), aspect="auto", cmap="viridis")
ax_matrix.set_xticks(range(len(sample.columns)))
ax_matrix.set_xticklabels([str(column) for column in sample.columns])
ax_matrix.set_ylabel("ROI voxels (subsampled)")
ax_matrix.set_title("Voxel feature matrix (raw)")
fig_matrix.colorbar(im, ax=ax_matrix, shrink=0.8, label="Intensity")
Path("out").mkdir(exist_ok=True)
fig_matrix.savefig("out/habitat_feature_routes_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

Fit the ``raw`` route and overlay habitats.



In [ ]:
raw_result = recipes.Study(spec=raw_spec).fit_predict(cohort)
print(raw_result.habitat_model.summary())
fig = plot_habitat_overlay(
    subject.image(MODALITIES[0]),
    raw_result.habitat_maps[0],
    title="habitats (raw)",
)
Path("out").mkdir(exist_ok=True)
fig.savefig("out/habitat_feature_routes_overlay.png", dpi=150, bbox_inches="tight")
plt.show()

``concat`` joins two ``raw`` extractors column-wise. The head() table
should show the same number of rows and more columns.



In [ ]:
m0, m1 = MODALITIES
concat_spec = HabitatSpec(
    name="route_concat",
    voxel_feature_extractor=Spec(
        "concat",
        {
            "extractors": [
                {"name": "raw", "params": {"modalities": [m0]}},
                {"name": "raw", "params": {"modalities": [m1]}},
            ],
        },
    ),
    supervoxelizer=Spec("kmeans", {"n_supervoxels": 8, "n_init": 3}),
    habitat_model_fitter=Spec(
        "kmeans",
        {"min_habitats": 2, "max_habitats": 3, "validation": "elbow", "n_init": 3},
    ),
    habitat_assigner=Spec("nearest_centroid"),
    habitat_features=(Spec("volume"), Spec("msi"), Spec("ith_score")),
    random_seed=11,
)
concat_field = (
    build_habitat_components(concat_spec)
    .pipeline(assigner=None)
    .voxel_feature_extractor(subject)
)
print("concat feature table:")
print(concat_field.feature_frame().head())
concat_field.feature_frame().head()
concat_result = recipes.Study(spec=concat_spec).fit_predict(cohort)
print(f"concat habitats: {concat_result.habitat_model.n_habitats}")